# Numerikus módszerek 8. gyakorlat – Papír-ceruza feladatok
## Progonka, iterációs módszerek: Jacobi, Gauss–Seidel, relaxált GS, Richardson

Minden feladatot **először oldd meg kézzel (papír-ceruza)**, majd ellenőrizd Pythonnal!

In [ ]:
import numpy as np
np.set_printoptions(precision=6, suppress=True)

def split_LDU(A):
    """A = L + D + U felbontás."""
    return np.tril(A, -1), np.diag(np.diag(A)), np.triu(A, 1)

def spectral_radius(B):
    return np.max(np.abs(np.linalg.eigvals(B)))

---
## 1. feladat – Progonka (Thomas-módszer) lépései

### Emlékeztető

Tridiagonális rendszer: $A = \mathrm{tridiag}(\beta_{i-1},\, \alpha_i,\, \gamma_i)$

**1. lépés (előre):**
$$f_1 = -\frac{\gamma_1}{\alpha_1},\quad g_1 = \frac{b_1}{\alpha_1}, \qquad
f_i = -\frac{\gamma_i}{\alpha_i + \beta_{i-1}f_{i-1}}, \quad g_i = \frac{b_i - \beta_{i-1}g_{i-1}}{\alpha_i + \beta_{i-1}f_{i-1}}$$
$$g_n = \frac{b_n - \beta_{n-1}g_{n-1}}{\alpha_n + \beta_{n-1}f_{n-1}}$$

**2. lépés (visszafelé):**
$x_n = g_n$, majd $x_i = f_i x_{i+1} + g_i$ ($i = n-1, \ldots, 1$)

---
### Feladat: oldd meg kézzel!

$$\begin{bmatrix} 3 & -1 & 0 \\ -1 & 3 & -1 \\ 0 & -1 & 3 \end{bmatrix} \begin{bmatrix}x_1\\x_2\\x_3\end{bmatrix} = \begin{bmatrix}2\\1\\2\end{bmatrix}$$

Tehát: $\alpha = (3, 3, 3)$, $\beta = (-1, -1)$, $\gamma = (-1, -1)$, $b = (2, 1, 2)$.

**Kézzel (töltsd ki!):**

1. lépés:
- $f_1 = -\gamma_1/\alpha_1 =$ ...
- $g_1 = b_1/\alpha_1 =$ ...
- $d_2 = \alpha_2 + \beta_1 f_1 =$ ...
- $f_2 = -\gamma_2/d_2 =$ ...
- $g_2 = (b_2 - \beta_1 g_1)/d_2 =$ ...
- $d_3 = \alpha_3 + \beta_2 f_2 =$ ...
- $g_3 = (b_3 - \beta_2 g_2)/d_3 =$ ...

2. lépés:
- $x_3 = g_3 =$ ...
- $x_2 = f_2 x_3 + g_2 =$ ...
- $x_1 = f_1 x_2 + g_1 =$ ...

In [ ]:
# Ellenőrzés
alpha = np.array([3., 3., 3.])
beta  = np.array([-1., -1.])
gamma = np.array([-1., -1.])
b     = np.array([2., 1., 2.])

# Progonka implementálása
n = len(alpha)
f = np.zeros(n); g = np.zeros(n)
f[0] = -gamma[0] / alpha[0]
g[0] = b[0] / alpha[0]
for i in range(1, n-1):
    d = alpha[i] + beta[i-1] * f[i-1]
    f[i] = -gamma[i] / d
    g[i] = (b[i] - beta[i-1]*g[i-1]) / d
d_n = alpha[n-1] + beta[n-2]*f[n-2]
g[n-1] = (b[n-1] - beta[n-2]*g[n-2]) / d_n

x = np.zeros(n)
x[n-1] = g[n-1]
for i in range(n-2, -1, -1):
    x[i] = f[i]*x[i+1] + g[i]

print('f =', f)
print('g =', g)
print('Megoldás x =', x)

A_full = np.diag(alpha) + np.diag(beta, -1) + np.diag(gamma, 1)
print('Ellenőrzés (Ax - b):', A_full @ x - b)

---
## 2. feladat – $A = L + D + U$ felbontás

### Emlékeztető

$L$ = az $A$ mátrix szigorúan alsó háromszögű része, $D$ = diagonális, $U$ = szigorúan felső háromszögű rész.

---
### Feladat: olvasd le az alábbi mátrix $L$, $D$, $U$ részeit!

$$A = \begin{bmatrix} 3 & -1 & 0 & 2 \\ -2 & 5 & 1 & 0 \\ 0 & 3 & -4 & -1 \\ 1 & 0 & 2 & 6 \end{bmatrix}$$

**Kézzel (töltsd ki!):**

$$L = \begin{bmatrix} \cdot & \cdot & \cdot & \cdot \\ \cdot & \cdot & \cdot & \cdot \\ \cdot & \cdot & \cdot & \cdot \\ \cdot & \cdot & \cdot & \cdot \end{bmatrix}, \quad
D = \begin{bmatrix} \cdot & & & \\ & \cdot & & \\ & & \cdot & \\ & & & \cdot \end{bmatrix}, \quad
U = \begin{bmatrix} \cdot & \cdot & \cdot & \cdot \\ \cdot & \cdot & \cdot & \cdot \\ \cdot & \cdot & \cdot & \cdot \\ \cdot & \cdot & \cdot & \cdot \end{bmatrix}$$

Ellenőrzés: $L + D + U = A$? (Igen / Nem)

In [ ]:
A2 = np.array([[ 3., -1.,  0.,  2.],
               [-2.,  5.,  1.,  0.],
               [ 0.,  3., -4., -1.],
               [ 1.,  0.,  2.,  6.]])

L2, D2, U2 = split_LDU(A2)
print('L =')
print(L2)
print('D (diagonális elemek):', np.diag(D2))
print('U =')
print(U2)
print('L + D + U == A:', np.allclose(L2 + D2 + U2, A2))

**Kérdések (töltsd ki!):**

1. Szigorúan diagonálisan domináns-e $A$ a soraira? (Ellenőrizd minden sorra: $|a_{ii}| > \sum_{j \ne i}|a_{ij}|$)
   - 1. sor: $|3| > |-1|+|0|+|2| = $ ... &nbsp; → ...
   - 2. sor: $|5| > |-2|+|1|+|0| = $ ... &nbsp; → ...
   - 3. sor: $|-4| > |0|+|3|+|-1| = $ ... &nbsp; → ...
   - 4. sor: $|6| > |1|+|0|+|2| = $ ... &nbsp; → ...
   
   Összességében: ...

---
## 3. feladat – Jacobi-iteráció kézzel

### Emlékeztető

$$x_i^{(k+1)} = \frac{1}{a_{ii}}\left(b_i - \sum_{j \ne i} a_{ij}\, x_j^{(k)}\right)$$

---
### Feladat: végezz el **3 Jacobi-lépést** $x^{(0)} = (0, 0, 0)^\top$ kezdőpontból!

$$A = \begin{bmatrix}5 & -1 & 0 \\ -1 & 5 & -1 \\ 0 & -1 & 5\end{bmatrix}, \quad b = \begin{bmatrix}4\\3\\4\end{bmatrix}$$

*(Megjegyzés: a pontos megoldás szimmetria alapján belátható.)*

**Kézzel (töltsd ki! Adj törtalakot is!):**

| $k$ | $x_1^{(k)}$ | $x_2^{(k)}$ | $x_3^{(k)}$ |
|---|---|---|---|
| 0 | 0 | 0 | 0 |
| 1 | ... | ... | ... |
| 2 | ... | ... | ... |
| 3 | ... | ... | ... |

Kérdések:
- Mi a pontos megoldás $x^*$? ...
- $\|x^{(3)} - x^*\|_\infty =$ ...

In [ ]:
A3 = np.array([[5., -1.,  0.],
               [-1., 5., -1.],
               [ 0., -1.,  5.]])
b3 = np.array([4., 3., 4.])

x = np.zeros(3)
d = np.diag(A3)
print(f'x^(0) = {x}')
for k in range(3):
    r = b3 - A3 @ x
    x = x + r / d
    print(f'x^({k+1}) = {x}  (törtalak: {np.array2string(x, formatter={"float": lambda v: f"{v}"})}')

x_exact = np.linalg.solve(A3, b3)
print(f'\nPontos megoldás x* = {x_exact}')
print(f'Hiba ||x^(3) - x*||_inf = {np.max(np.abs(x - x_exact)):.6f}')

---
## 4. feladat – S.d.d. ellenőrzés és $\|B_J\|_\infty$ kiszámítása

### Emlékeztető

**Szigorúan diagonálisan domináns (s.d.d.):** $|a_{ii}| > \sum_{j \ne i} |a_{ij}|$ minden $i$-re.

**Jacobi átmenet mátrix:** $B_J = -D^{-1}(L+U)$, azaz $(B_J)_{ij} = -a_{ij}/a_{ii}$ ha $i \ne j$, 0 ha $i = j$.

**$\infty$-norma:** $\|B_J\|_\infty = \max_i \sum_{j} |(B_J)_{ij}| = \max_i \frac{1}{|a_{ii}|} \sum_{j \ne i} |a_{ij}|$

Ha s.d.d., akkor $\|B_J\|_\infty < 1$, tehát Jacobi konvergens.

---
### Feladat:

Adott:
$$A = \begin{bmatrix} 4 & 1 & -1 \\ 2 & 6 & 1 \\ -1 & 2 & 5 \end{bmatrix}$$

a) Ellenőrizd, hogy $A$ s.d.d.-e!

b) Írd fel $B_J$ mátrixot (törtalakok!)!

c) Számítsd ki $\|B_J\|_\infty$-t!

**Kézzel (töltsd ki!):**

**a) S.d.d. ellenőrzés:**
- 1. sor: $|4| > |1| + |-1| = $ ... &nbsp;→ ...
- 2. sor: $|6| > |2| + |1|  = $ ... &nbsp;→ ...
- 3. sor: $|5| > |-1| + |2| = $ ... &nbsp;→ ...
- $A$ s.d.d.: ...

**b) $B_J$ mátrix:** $(B_J)_{ij} = -a_{ij}/a_{ii}$ ($j \ne i$), 0 az átlón:

$$B_J = \begin{bmatrix} 0 & \cdots & \cdots \\ \cdots & 0 & \cdots \\ \cdots & \cdots & 0 \end{bmatrix}$$

**c) $\|B_J\|_\infty$:**
- 1. sor összege: ...
- 2. sor összege: ...
- 3. sor összege: ...
- $\|B_J\|_\infty = $ ...

In [ ]:
A4 = np.array([[ 4.,  1., -1.],
               [ 2.,  6.,  1.],
               [-1.,  2.,  5.]])

d4 = np.diag(A4)

# S.d.d. ellenőrzés
for i in range(3):
    offdiag_sum = sum(abs(A4[i,j]) for j in range(3) if j != i)
    print(f'Sor {i+1}: |{A4[i,i]:.0f}| vs {offdiag_sum:.0f}  → s.d.d.: {abs(A4[i,i]) > offdiag_sum}')

# B_J
L4, D4, U4 = split_LDU(A4)
B_J4 = -np.linalg.inv(D4) @ (L4 + U4)
print('\nB_J =')
print(B_J4)
print(f'||B_J||_inf = {np.linalg.norm(B_J4, ord=np.inf):.6f}')
print(f'rho(B_J)    = {spectral_radius(B_J4):.6f}')

---
## 5. feladat – Gauss–Seidel iteráció kézzel

### Emlékeztető

A Gauss–Seidel az $i$-edik lépésben a frissen számított $x_j^{(k+1)}$ értékeket azonnal felhasználja:

$$x_i^{(k+1)} = \frac{1}{a_{ii}}\left(b_i - \sum_{j=1}^{i-1} a_{ij}\, x_j^{(k+1)} - \sum_{j=i+1}^{n} a_{ij}\, x_j^{(k)}\right)$$

---
### Feladat: végezz el **3 Gauss–Seidel-lépést** ugyanarra a rendszerre!

$$A = \begin{bmatrix}5 & -1 & 0 \\ -1 & 5 & -1 \\ 0 & -1 & 5\end{bmatrix}, \quad b = \begin{bmatrix}4\\3\\4\end{bmatrix}, \quad x^{(0)} = (0,0,0)^\top$$

**Kézzel (töltsd ki!):**

**1. lépés ($k=0 \to 1$):**
- $x_1^{(1)} = (4 - (-1)\cdot x_2^{(0)} - 0\cdot x_3^{(0)})/5 = $ ...
- $x_2^{(1)} = (3 - (-1)\cdot x_1^{(1)} - (-1)\cdot x_3^{(0)})/5 = $ ...
- $x_3^{(1)} = (4 - 0\cdot x_1^{(1)} - (-1)\cdot x_2^{(1)})/5 = $ ...

**2. lépés ($k=1 \to 2$):**
- $x_1^{(2)} = $ ...
- $x_2^{(2)} = $ ...
- $x_3^{(2)} = $ ...

**3. lépés ($k=2 \to 3$):**
- $x_1^{(3)} = $ ...
- $x_2^{(3)} = $ ...
- $x_3^{(3)} = $ ...

**Összehasonlítás a 3. feladattal (Jacobi):**
- $\|x^{(3)}_{\text{GS}} - x^*\|_\infty \approx$ ...
- $\|x^{(3)}_{\text{J}} - x^*\|_\infty \approx$ ...
- Melyik konvergál gyorsabban? ...

In [ ]:
x = np.zeros(3)
print(f'x^(0) = {x}')
for k in range(3):
    r = b3 - A3 @ x
    n = 3
    L3, D3, U3 = split_LDU(A3)
    LD3 = L3 + D3
    s = np.zeros(n)
    for i in range(n):
        s[i] = (r[i] - LD3[i, :i] @ s[:i]) / LD3[i, i]
    x = x + s
    print(f'x^({k+1}) = {x}')

print(f'\nPontos megoldás x* = {x_exact}')
print(f'Hiba ||x^(3)_GS - x*||_inf = {np.max(np.abs(x - x_exact)):.6f}')

---
## 6. feladat – Kontrakció, fixpont, hibabecslés

### Emlékeztető (Banach-fixponttétel)

Ha $\varphi(x) = Bx + c$ kontrakció $q$ együtthatóval, akkor:
- **A priori:** $\|x^{(k)} - x^*\| \le q^k \|x^{(0)} - x^*\|$
- **A posteriori:** $\|x^{(k)} - x^*\| \le \dfrac{q^k}{1-q} \|x^{(1)} - x^{(0)}\|$

Az iteráció akkor konvergens (minden $x^{(0)}$-ra), ha $\varrho(B) < 1$.

---
### Feladat:

Adott az iteráció: $x^{(k+1)} = Bx^{(k)} + c$, ahol

$$B = \begin{bmatrix} 0 & \tfrac{1}{2} \\ \tfrac{1}{2} & 0 \end{bmatrix}, \quad c = \begin{bmatrix} 1 \\ 1 \end{bmatrix}$$

**a)** Számítsd ki $\varrho(B)$-t a karakterisztikus polinom segítségével!

**b)** Határozd meg a fixpontot $x^* = (I - B)^{-1}c$ alapján (kézi mátrixinvertálással)!

**c)** $x^{(0)} = (0, 0)^\top$-ból indulva adj a priori felső korlátot $\|x^{(4)} - x^*\|_\infty$-re!

**d)** Számítsd ki $x^{(1)}$-et, majd adj a posteriori felső korlátot $\|x^{(4)} - x^*\|_\infty$-re!

**Kézzel (töltsd ki!):**

**a) Spektrálsugár:**
$$\det(B - \lambda I) = \det\begin{bmatrix}-\lambda & 1/2 \\ 1/2 & -\lambda\end{bmatrix} = \lambda^2 - \frac{1}{4} = 0 \implies \lambda_{1,2} = \pm \ldots$$
$$\varrho(B) = \ldots$$

**b) Fixpont:**
$$I - B = \begin{bmatrix} 1 & -1/2 \\ -1/2 & 1 \end{bmatrix}, \quad \det(I-B) = 1 - \frac{1}{4} = \ldots$$
$$(I-B)^{-1} = \frac{1}{\det} \begin{bmatrix} 1 & 1/2 \\ 1/2 & 1 \end{bmatrix} = \begin{bmatrix} \ldots & \ldots \\ \ldots & \ldots \end{bmatrix}$$
$$x^* = (I-B)^{-1}c = \ldots$$

**c) A priori becslés ($k=4$):**
$$\|x^{(0)} - x^*\|_\infty = \max(|0 - x^*_1|, |0 - x^*_2|) = \ldots$$
$$\|x^{(4)} - x^*\|_\infty \le q^4 \cdot \|x^{(0)} - x^*\|_\infty = \left(\frac{1}{2}\right)^4 \cdot \ldots = \ldots$$

**d) A posteriori becslés:**
$$x^{(1)} = B \cdot (0,0)^\top + c = \ldots$$
$$\|x^{(1)} - x^{(0)}\|_\infty = \ldots$$
$$\|x^{(4)} - x^*\|_\infty \le \frac{q^4}{1-q} \cdot \|x^{(1)} - x^{(0)}\|_\infty = \frac{(1/2)^4}{1/2} \cdot \ldots = \ldots$$

In [ ]:
B6 = np.array([[0., 0.5], [0.5, 0.]])
c6 = np.array([1., 1.])

# Spektrálsugár
rho6 = spectral_radius(B6)
print(f'rho(B) = {rho6}  (= 1/2: {np.isclose(rho6, 0.5)})')

# Fixpont
x_star6 = np.linalg.solve(np.eye(2) - B6, c6)
print(f'x* = {x_star6}')

# Iteráció
x6 = np.zeros(2)
for k in range(5):
    x6 = B6 @ x6 + c6
    err = np.linalg.norm(x6 - x_star6, ord=np.inf)
    print(f'x^({k+1}) = {x6},   ||x^({k+1}) - x*||_inf = {err:.6f}')

# Hibabecslések
q = 0.5
err0 = np.linalg.norm(np.zeros(2) - x_star6, ord=np.inf)
apriori = q**4 * err0
x1_6 = B6 @ np.zeros(2) + c6
aposteriori = q**4 / (1-q) * np.linalg.norm(x1_6 - np.zeros(2), ord=np.inf)
print(f'\nA priori k=4 korlát:     {apriori:.6f}')
print(f'A posteriori k=4 korlát: {aposteriori:.6f}')
print(f'Valódi hiba k=4:         {np.linalg.norm(B6@B6@B6@B6@np.zeros(2) + B6@B6@B6@c6 + B6@B6@c6 + B6@c6 + c6 - x_star6, ord=np.inf):.6f}')

---
## 7. feladat – Optimális relaxációs paraméter

### Emlékeztető

Szimmetrikus, pozitív definit, tridiagonális esetben a relaxált Gauss–Seidel $S(\omega)$ optimális paramétere:
$$\omega_0 = \frac{2}{1 + \sqrt{1 - \varrho(B_J)^2}}$$

Az optimális spektrálsugár: $\varrho(B_{S(\omega_0)}) = \omega_0 - 1$

Szükséges konvergencia-feltétel: $0 < \omega < 2$.

---
### Feladat: számítsd ki $\omega_0$-t és $\varrho(B_{S(\omega_0)})$-t az alábbi esetekre!

**a)** $\varrho(B_J) = 0{,}6$

**b)** $\varrho(B_J) = 0{,}8$

**c)** $\varrho(B_J) = \dfrac{\sqrt{2}}{2}$ (azaz $\approx 0{,}707$)

*(c-nél elég közelítő érték is.)*

**Kézzel (töltsd ki!):**

**a)** $\varrho(B_J) = 0{,}6$:
$$\omega_0 = \frac{2}{1 + \sqrt{1 - 0{,}36}} = \frac{2}{1 + \sqrt{0{,}64}} = \frac{2}{1 + 0{,}8} = \ldots$$
$$\varrho(B_{S(\omega_0)}) = \omega_0 - 1 = \ldots$$

**b)** $\varrho(B_J) = 0{,}8$:
$$\omega_0 = \frac{2}{1 + \sqrt{1 - 0{,}64}} = \frac{2}{1 + \sqrt{\ldots}} = \ldots$$
$$\varrho(B_{S(\omega_0)}) = \omega_0 - 1 = \ldots$$

**c)** $\varrho(B_J) = \dfrac{\sqrt{2}}{2}$:
$$1 - \varrho(B_J)^2 = 1 - \frac{1}{2} = \frac{1}{2}, \quad \sqrt{\frac{1}{2}} = \ldots$$
$$\omega_0 = \frac{2}{1 + \ldots} = \ldots, \quad \varrho(B_{S(\omega_0)}) = \ldots$$

**Összehasonlítás:** Mit mutatnak ezek az értékek? Hogyan változik a konvergencia sebessége $\varrho(B_J)$ növekedésével?

...

In [ ]:
for rho_J, label in [(0.6, 'a'), (0.8, 'b'), (np.sqrt(2)/2, 'c')]:
    omega0 = 2 / (1 + np.sqrt(1 - rho_J**2))
    rho_opt = omega0 - 1
    print(f'({label}) rho(B_J)={rho_J:.4f}:  omega_0 = {omega0:.6f},  rho(B_S(omega_0)) = {rho_opt:.6f}')

# Szemléltetés: hány iteráció kell tol=1e-8-hoz?
import math
tol = 1e-8
print()
for rho_J, label in [(0.6, 'a'), (0.8, 'b'), (np.sqrt(2)/2, 'c')]:
    rho_GS = rho_J**2
    omega0 = 2 / (1 + np.sqrt(1 - rho_J**2))
    rho_opt = omega0 - 1
    k_J  = math.ceil(math.log(tol) / math.log(rho_J))
    k_GS = math.ceil(math.log(tol) / math.log(rho_GS))
    k_S  = math.ceil(math.log(tol) / math.log(rho_opt)) if rho_opt > 0 else 1
    print(f'({label}) Jacobi≈{k_J} it., Gauss-Seidel≈{k_GS} it., Relaxált GS≈{k_S} it.')

---
## 8. feladat – Richardson-iteráció: optimális paraméter

### Emlékeztető

Szimmetrikus, pozitív definit $A$ esetén ($\lambda_1 = m \le \cdots \le \lambda_n = M$):
- Konvergencia: $p \in \left(0,\, \dfrac{2}{M}\right)$
- Optimális: $p_0 = \dfrac{2}{M+m}$, kontrakció: $q = \dfrac{M-m}{M+m}$

A sajátértékeket a **karakterisztikus polinom** gyökeivel kapjuk: $\det(A - \lambda I) = 0$

---
### Feladat:

$$A = \begin{bmatrix} 5 & -2 \\ -2 & 5 \end{bmatrix}, \quad b = \begin{bmatrix} 3 \\ 3 \end{bmatrix}$$

a) Határozd meg $A$ sajátértékeit a karakterisztikus polinom segítségével!

b) Számítsd ki az optimális $p_0$-t és a kontrakciós $q$ együtthatót!

c) Adj meg egy $p$ értéket, amelyre az iteráció **nem** konvergens!

d) Végezz el **2 Richardson-lépést** $x^{(0)} = (0, 0)^\top$-ból, $p = p_0$-val!

**Kézzel (töltsd ki!):**

**a) Sajátértékek:**
$$\det(A - \lambda I) = \det\begin{bmatrix}5-\lambda & -2 \\ -2 & 5-\lambda\end{bmatrix} = (5-\lambda)^2 - 4 = 0$$
$$(5-\lambda)^2 = 4 \implies 5-\lambda = \pm 2 \implies \lambda_1 = \ldots, \quad \lambda_2 = \ldots$$

**b) Optimális paraméter:**
- $m = \ldots$, $M = \ldots$
- Konvergens $p$ tartomány: $p \in (0,\, \ldots)$
- $p_0 = \dfrac{2}{M + m} = \ldots$
- $q = \dfrac{M - m}{M + m} = \ldots$

**c) Nem konvergens $p$:** ...</n

**d) Richardson-lépések ($p = p_0$):**

$x^{(k+1)} = x^{(k)} + p_0\, r^{(k)}, \quad r^{(k)} = b - Ax^{(k)}$

- $r^{(0)} = b - A x^{(0)} = \ldots$
- $x^{(1)} = x^{(0)} + p_0 r^{(0)} = \ldots$
- $r^{(1)} = r^{(0)} - A(p_0 r^{(0)}) = \ldots$
- $x^{(2)} = x^{(1)} + p_0 r^{(1)} = \ldots$

Pontos megoldás: $x^* = $ ...

In [ ]:
A8 = np.array([[5., -2.], [-2., 5.]])
b8 = np.array([3., 3.])

# Sajátértékek
eigvals8 = np.linalg.eigvalsh(A8)
m8, M8 = eigvals8.min(), eigvals8.max()
print(f'Sajátértékek: m = {m8:.0f}, M = {M8:.0f}')

p0 = 2 / (M8 + m8)
q8 = (M8 - m8) / (M8 + m8)
print(f'p0 = {p0},  q = {q8}')
print(f'Konvergens p tartomány: (0, {2/M8})')

# Richardson lépések
x8 = np.zeros(2)
x_exact8 = np.linalg.solve(A8, b8)
print(f'\nPontos megoldás: {x_exact8}')
print(f'x^(0) = {x8}')
for k in range(5):
    r8 = b8 - A8 @ x8
    x8 = x8 + p0 * r8
    print(f'x^({k+1}) = {x8},   hiba = {np.max(np.abs(x8 - x_exact8)):.6f}')

---
## Összefoglaló táblázat – töltsd ki!

| Módszer | Átmenet mátrix | Konvergencia feltétele | Optimális paraméter |
|---|---|---|---|
| Progonka | – (direkt) | tridiagonális LER esetén alkalmazható | – |
| Jacobi $J(1)$ | $-D^{-1}(L+U)$ | s.d.d. $\Rightarrow$ ... | – |
| Csillapított $J(\omega)$ | ... | $J(1)$ konvergens + $0<\omega<1$ | – |
| Gauss–Seidel $S(1)$ | ... | s.d.d. vagy szimm.+pozdef. | – |
| Relaxált $S(\omega)$ | ... | szükséges: ... | $\omega_0 = \dfrac{2}{1+\sqrt{1-\varrho(B_J)^2}}$ |
| Richardson $R(p)$ | $(I-pA)$ | szimm.+pozdef.: $p \in (0, \ldots)$ | $p_0 = \dfrac{2}{M+m}$ |

**Tridiagonális eset:** $\varrho(B_S) = $ ...